# Qwen 3.6 35B-A3B - Kaggle server (Gradio UI + OpenAI API via ngrok)
**Before running:** Settings -> Accelerator = *GPU T4 x2*, Internet = *On*; Add-ons -> Secrets -> attach `NGROK_AUTHTOKEN`, `NGROK_DOMAIN`, `API_KEY`; Input -> Add the Qwen3.6-35B-A3B model.
Then **Run All**. The last cells print the public URL.

In [1]:
!nvidia-smi

Sat Nov 14 21:28:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Kaggle config (replaces the Google Drive mount)
import os, glob

assert os.path.isdir("/kaggle/working"), "This notebook is meant to run on Kaggle."

MODEL_ID = "qwen3.6-35b-a3b"
MODEL_CONTEXT_TOKENS = 42000          # start value; the loader steps down automatically if VRAM is short
MODEL_FILE_HINT = "Q4_K_P"            # preferred quant if the attached model folder has several .gguf files

WORK_DIR = "/kaggle/working"
print("Working dir:", WORK_DIR)
print("Attached inputs:")
for p in sorted(glob.glob("/kaggle/input/*")):
    print("  ", p)


Working dir: /kaggle/working
Attached inputs:
   /kaggle/input/models


In [3]:
# 2. Install deps + CUDA build of llama-cpp-python (Kaggle has no persistent cache, so this runs each session)
import os, sys, re, subprocess

def sh(cmd, timeout=None):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

r = sh(f"{sys.executable} -m pip -q install gradio pyngrok pypdf")
if r.returncode:
    print(r.stderr[-2000:])

def have_gpu_llama():
    if "llama_cpp" not in sys.modules:
        return False
    import llama_cpp
    return bool(llama_cpp.llama_supports_gpu_offload())

if have_gpu_llama():
    print("llama-cpp-python with CUDA already loaded in this session - skipping install.")
else:
    nvcc = sh("nvcc --version").stdout
    m = re.search(r"release (\d+)\.(\d+)", nvcc)
    detected = f"cu{m.group(1)}{m.group(2)}" if m else None
    print("Python:", sys.version.split()[0], "| CUDA toolkit tag:", detected)

    tags = []
    for t in (detected, "cu125", "cu124", "cu123", "cu122", "cu121"):
        if t and t not in tags:
            tags.append(t)

    installed = False
    for tag in tags:
        print(f"Trying prebuilt wheel index {tag} ...")
        r = sh(f"{sys.executable} -m pip -q install llama-cpp-python --only-binary=:all: --force-reinstall --no-deps "
               f"--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{tag}")
        if r.returncode == 0:
            installed = True
            print("  installed from", tag)
            break
        print("  no wheel:", r.stderr.strip().splitlines()[-1][:160] if r.stderr.strip() else "unknown error")

    if not installed:
        print("No prebuilt wheel matched - building from source for T4 (sm_75). This can take 10-25 minutes...")
        env = ('CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75" FORCE_CMAKE=1 ')
        r = sh(env + f"{sys.executable} -m pip install llama-cpp-python --no-cache-dir --force-reinstall --no-deps", timeout=3000)
        if r.returncode:
            raise RuntimeError("llama-cpp-python build failed:\n" + r.stderr[-3000:])

    r = sh(f"{sys.executable} -m pip -q install numpy typing-extensions diskcache jinja2")

import llama_cpp
print("llama-cpp-python", llama_cpp.__version__, "| GPU offload:", llama_cpp.llama_supports_gpu_offload())
if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError("llama-cpp-python has no CUDA support. Check Settings -> Accelerator = GPU T4 x2, then Restart & Run All.")


Python: 3.13.16 | CUDA toolkit tag: cu130
Trying prebuilt wheel index cu130 ...
  installed from cu130


ggml_cuda_init: found 2 CUDA devices (Total VRAM: 29823 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14911 MiB
  Device 1: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14911 MiB


llama-cpp-python 0.3.36 | GPU offload: True


In [4]:
# 3. Model: read straight from the attached Kaggle model (no download, no copy)
import os, glob, re

MIN_BYTES = 1_000_000_000

def find_model(hint=MODEL_FILE_HINT):
    files = [p for p in glob.glob("/kaggle/input/**/*.gguf", recursive=True) if os.path.isfile(p)]  # skip folders that are named *.gguf
    if not files:
        raise FileNotFoundError(
            "No .gguf found under /kaggle/input. Open the right sidebar -> Input -> Add Input -> Models, "
            "add Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive, then re-run.")
    # split GGUFs: llama.cpp wants the first shard
    firsts = [p for p in files if re.search(r"-0*1-of-\d+\.gguf$", p)]
    others = [p for p in files if not re.search(r"-\d+-of-\d+\.gguf$", p)]
    pool = firsts + others or files
    hinted = [p for p in pool if hint.lower() in os.path.basename(p).lower()]
    return sorted(hinted or pool)[0], files

MODEL_PATH, _all = find_model()
if len(_all) > 1:
    print("GGUF files found:")
    for p in _all:
        print("  ", p)

size_gb = os.path.getsize(MODEL_PATH) / 1024**3
assert os.path.getsize(MODEL_PATH) > MIN_BYTES, "Model file looks truncated."
print("MODEL_PATH:", MODEL_PATH, f"({size_gb:.2f} GB)")
if size_gb > 27:
    print("WARNING: model is >27 GB; it may not leave room for the KV cache on 2x T4. Use a smaller quant.")


MODEL_PATH: /kaggle/input/models/yuvvraaj7/qwen3-6-35b-a3b-hauhaucs-aggressive-q4kp/gguf/qwen3.6-35b-a3b-uncensored-hauhaucs-aggressive-q4_k_p.gguf/1/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_P.gguf (21.82 GB)


In [5]:
# 4. Chat DB (lives in /kaggle/working; see note below about persistence)
import os, glob, shutil, sqlite3

CHAT_DB = "/kaggle/working/chat_history.sqlite3"

# Optional: if you attach a Dataset that contains an earlier chat_history.sqlite3, restore it once.
if not os.path.exists(CHAT_DB):
    old = glob.glob("/kaggle/input/**/chat_history.sqlite3", recursive=True)
    if old:
        shutil.copy(old[0], CHAT_DB)
        print("Restored previous chat history from:", old[0])

with sqlite3.connect(CHAT_DB, timeout=30) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
print("Chat database:", CHAT_DB)
# NOTE: /kaggle/working is wiped when the session ends. To keep history, click "Save Version" (output is kept),
# or download chat_history.sqlite3 and attach it later as a Dataset - it is restored automatically above.


Chat database: /kaggle/working/chat_history.sqlite3


In [6]:
# 5. Secrets from Kaggle (Add-ons -> Secrets; each one must be attached/checked for THIS notebook)
import os
from kaggle_secrets import UserSecretsClient

_secrets_client = UserSecretsClient()

def get_secret(name, default=None):
    """Never crashes: returns default if the secret is missing or not attached to this notebook."""
    try:
        v = _secrets_client.get_secret(name)
    except Exception as e:
        print(f"[secrets] {name}: {type(e).__name__}: {str(e)[:120]}")
        v = None
    return v or os.environ.get(name) or default

ngrok_token  = get_secret("NGROK_AUTHTOKEN")
ngrok_domain = get_secret("NGROK_DOMAIN")
API_KEY      = get_secret("API_KEY", "sk-gemma4-balanced-custom")

missing = [n for n, v in (("NGROK_AUTHTOKEN", ngrok_token), ("NGROK_DOMAIN", ngrok_domain)) if not v]
if missing:
    raise RuntimeError(
        f"Missing secrets: {missing}. Open Add-ons -> Secrets, create them with these exact names "
        "(case-sensitive) and tick them so they are attached to this notebook. Also turn Internet On.")

def _clean(v):
    return v.strip().strip('"').strip("'").strip() if isinstance(v, str) else v

ngrok_token = _clean(ngrok_token)
API_KEY = _clean(API_KEY)
print(f"[check] NGROK_AUTHTOKEN: length={len(ngrok_token)}, starts with {ngrok_token[:3]!r}, ends with {ngrok_token[-2:]!r}, has_space={' ' in ngrok_token}")

ngrok_domain = ngrok_domain.strip().rstrip("/").removeprefix("https://").removeprefix("http://").split("/")[0]
print("Secrets loaded: NGROK_AUTHTOKEN, NGROK_DOMAIN, API_KEY(" + ("custom" if API_KEY != "sk-gemma4-balanced-custom" else "fallback") + ")")


[check] NGROK_AUTHTOKEN: length=49, starts with '3M6', ends with 'g6', has_space=False
Secrets loaded: NGROK_AUTHTOKEN, NGROK_DOMAIN, API_KEY(custom)


In [7]:
import os
import sqlite3
import uuid
import time
import gradio as gr

SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Give clear, accurate and thoughtful answers."
)

# ---------------- Database helpers ----------------

def get_conversations():
    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT conversation_id, MAX(id) AS last_id
            FROM messages
            GROUP BY conversation_id
            ORDER BY last_id DESC
        """).fetchall()
    return [row[0] for row in rows]


def get_messages(cid):
    if not cid:
        return []

    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT role, content
            FROM messages
            WHERE conversation_id = ?
            ORDER BY id
        """, (cid,)).fetchall()

    return [
        {"role": role, "content": content}
        for role, content in rows
        if role in ("user", "assistant")
    ]


def save_message(cid, role, content):
    with sqlite3.connect(CHAT_DB) as con:
        con.execute("""
            INSERT INTO messages (conversation_id, role, content)
            VALUES (?, ?, ?)
        """, (cid, role, content))


# ---------------- Attachment reader ----------------

def read_uploaded_file(file_path):
    filename = os.path.basename(file_path)
    extension = os.path.splitext(filename)[1].lower()

    if extension == ".txt":
        with open(file_path, "r", encoding="utf-8-sig") as uploaded_file:
            text = uploaded_file.read()
        if not text.strip():
            raise ValueError("The uploaded text file is empty.")
        return text

    if extension == ".pdf":
        from pypdf import PdfReader

        page_texts = []
        for page in PdfReader(file_path).pages:
            page_text = page.extract_text()
            if page_text and page_text.strip():
                page_texts.append(page_text.strip())
        text = "\n\n".join(page_texts)
        if not text:
            raise ValueError(
                "No selectable text was found in the PDF. Scanned PDFs require OCR."
            )
        return text

    raise ValueError("Only .txt and .pdf files are supported.")


# ---------------- Chat actions ----------------

def new_conversation():
    cid = str(uuid.uuid4())[:8]
    choices = get_conversations()

    if cid not in choices:
        choices.insert(0, cid)

    return cid, [], gr.update(choices=choices, value=cid), None


def load_conversation(cid):
    if not cid:
        return "", [], None
    return cid, get_messages(cid), None


def chat(message, cid, file_path=None):
    received_at = time.perf_counter()
    message = (message or "").strip()
    cid = (cid or "").strip()
    print(
        f"[chat] Prompt received ({len(message)} characters); "
        f"conversation={cid or 'new'}, attachment={'yes' if file_path else 'no'}.",
        flush=True,
    )

    if file_path:
        try:
            print("[chat] Reading uploaded attachment...", flush=True)
            file_text = read_uploaded_file(file_path)
        except Exception as e:
            print(f"[chat] Attachment read failed: {type(e).__name__}: {e}", flush=True)
            raise gr.Error(
                f"Unable to read uploaded file ({type(e).__name__}): {e}"
            ) from e

        attachment_prompt = (
            f"Contents of uploaded file ({os.path.basename(file_path)}):\n{file_text}"
        )
        message = "\n\n".join(
            part for part in (message, attachment_prompt) if part
        )
        print(
            f"[chat] Attachment read ({len(file_text)} characters).",
            flush=True,
        )

    if not message:
        print("[chat] Empty prompt; skipping generation.", flush=True)
        yield "", get_messages(cid), cid, gr.update(
            choices=get_conversations(), value=cid or None
        ), None
        return

    if not cid:
        cid = str(uuid.uuid4())[:8]

    save_message(cid, "user", message)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + get_messages(cid)
    prompt_chars = sum(len(item.get("content") or "") for item in messages)
    print(
        f"[chat] Prompt prepared: {len(messages)} messages, "
        f"{prompt_chars} characters. Formatting with thinking disabled...",
        flush=True,
    )

    choices = get_conversations()
    if cid not in choices:
        choices.insert(0, cid)

    visible_history = get_messages(cid)
    yield (
        "",
        visible_history + [{"role": "assistant", "content": "Generating..."}],
        cid,
        gr.update(choices=choices, value=cid),
        None,
    )

    answer = ""
    first_token_logged = False
    try:
        prompt_text = format_chat_prompt(messages, enable_thinking=False)
        prompt_tokens = llm.tokenize(
            prompt_text.encode("utf-8"), add_bos=False, special=True
        )
        prompt_tokens, max_output_tokens = fit_prompt_to_context(prompt_tokens)

        print(
            f"[chat] Prompt ready ({len(prompt_tokens)} tokens). "
            "Generating response without reasoning...",
            flush=True,
        )
        generation_started = time.perf_counter()
        for chunk in llm.create_completion(
            prompt=prompt_tokens,
            max_tokens=max_output_tokens,
            temperature=0.6,
            top_p=0.9,
            stop=["<|im_end|>", "<|endoftext|>"],
            stream=True,
        ):
            token_text = chunk["choices"][0].get("text", "")
            if not isinstance(token_text, str) or not token_text:
                continue

            if not first_token_logged:
                first_token_logged = True
                print(
                    f"[chat] First response token after "
                    f"{time.perf_counter() - generation_started:.1f}s.",
                    flush=True,
                )

            answer += token_text
            yield (
                "",
                visible_history + [{"role": "assistant", "content": answer}],
                cid,
                gr.update(choices=choices, value=cid),
                None,
            )

        if not answer.strip():
            answer = "(The model returned an empty response.)"
        save_message(cid, "assistant", answer)
        print(
            f"[chat] Generation complete in "
            f"{time.perf_counter() - generation_started:.1f}s "
            f"({len(answer)} response characters).",
            flush=True,
        )
        final_history = get_messages(cid)
    except Exception as e:
        answer = f"Model error: {type(e).__name__}: {e}"
        print(f"[chat] Model generation failed: {answer}", flush=True)
        final_history = visible_history + [{"role": "assistant", "content": answer}]

    print(
        f"[chat] Request finished in {time.perf_counter() - received_at:.1f}s.",
        flush=True,
    )
    yield (
        "",
        final_history,
        cid,
        gr.update(choices=choices, value=cid),
        None,
    )


# ---------------- Gradio interface ----------------

with gr.Blocks(title="My AI Assistant") as demo:
    gr.Markdown("# My AI Assistant")
    gr.Markdown("Qwen 3.6 35B-A3B · Chat history saved in SQLite")

    with gr.Row():
        cid = gr.Textbox(
            label="Conversation ID",
            placeholder="Create a new chat or select a saved one",
            scale=3
        )
        new_btn = gr.Button("New chat")

    saved_chats = gr.Dropdown(
        label="Saved conversations",
        choices=get_conversations(),
        interactive=True
    )

    chatbot_ui = gr.Chatbot(
    height=500,
    label="Chat"
    )

    message = gr.Textbox(
        label="Your message",
        placeholder="Message your AI...",
        lines=2
    )

    attachment = gr.File(
        label="Attach a .txt or .pdf file",
        file_types=[".txt", ".pdf"],
        type="filepath",
    )

    send_btn = gr.Button("Send", variant="primary")

    new_btn.click(
        new_conversation,
        outputs=[cid, chatbot_ui, saved_chats, attachment]
    )

    saved_chats.change(
        load_conversation,
        inputs=saved_chats,
        outputs=[cid, chatbot_ui, attachment]
    )

    send_btn.click(
        chat,
        inputs=[message, cid, attachment],
        outputs=[message, chatbot_ui, cid, saved_chats, attachment]
    )

    message.submit(
        chat,
        inputs=[message, cid, attachment],
        outputs=[message, chatbot_ui, cid, saved_chats, attachment]
    )

    demo.load(
        lambda: ("", [], None),
        outputs=[cid, chatbot_ui, attachment]
    )

print("Chat functions restored:", callable(chat))
print("Gradio interface restored:", demo is not None)

Chat functions restored: True
Gradio interface restored: True


In [8]:
import gc, llama_cpp
from llama_cpp import Llama

if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError("llama-cpp-python was installed without CUDA support. Re-run the install cell.")

# Free any previous model first (re-running this cell otherwise keeps old VRAM allocated)
try:
    del llm
except NameError:
    pass
gc.collect()

def load_llm(n_ctx):
    kwargs = dict(
        model_path=MODEL_PATH,
        n_gpu_layers=-1,
        tensor_split=[0.5, 0.5],           # split layers evenly across the two T4s
        main_gpu=0,
        n_ctx=n_ctx,
        n_batch=512,
        n_ubatch=256,                      # smaller compute buffer
        flash_attn=True,
        type_k=llama_cpp.GGML_TYPE_Q8_0,   # q8 KV cache: roughly half the f16 size
        type_v=llama_cpp.GGML_TYPE_Q8_0,
        verbose=False,
    )
    try:
        return Llama(swa_full=False, **kwargs)   # key fix: sliding-window layers keep only their window
    except TypeError:
        print("This llama-cpp-python has no swa_full option; trying without it.")
        return Llama(**kwargs)

llm, last_err = None, None
for ctx in (MODEL_CONTEXT_TOKENS, 16384, 12288, 8192, 4096):
    try:
        print(f"Trying n_ctx={ctx:,} ...")
        llm = load_llm(ctx)
        MODEL_CONTEXT_TOKENS = ctx
        break
    except Exception as e:
        last_err = e
        print(f"  failed: {type(e).__name__}: {e}")
        gc.collect()

if llm is None:
    raise RuntimeError(f"Could not create a context even at 4,096 tokens: {last_err}")

print(f"Model loaded on GPU with {MODEL_CONTEXT_TOKENS:,}-token context.")

def fit_prompt_to_context(prompt_tokens, prompt_text=None, requested_max_tokens=None):
    max_ctx = llm.n_ctx() if callable(llm.n_ctx) else llm.n_ctx
    try:
        requested = int(requested_max_tokens) if requested_max_tokens is not None else 2048
    except (TypeError, ValueError):
        requested = 2048
    if requested < 1:
        requested = 2048

    max_output = min(requested, max_ctx - 1)
    min_output = min(1024, max_output, max_ctx - 1)
    prompt_limit = max_ctx - min_output
    prefix_len = suffix_len = 0

    if prompt_text:
        user_markers = list(re.finditer(r"<\|im_start\|>user\n", prompt_text))
        if user_markers:
            latest_request = None
            for marker in reversed(user_markers):
                if not prompt_text[marker.end():].lstrip().startswith("<tool_response>"):
                    latest_request = marker.start()
                    break
            if latest_request is None:
                latest_request = user_markers[-1].start()
            prefix_text = prompt_text[:user_markers[0].start()]
            suffix_text = prompt_text[latest_request:]
            prefix_len = len(llm.tokenize(prefix_text.encode("utf-8"), add_bos=False, special=True))
            suffix_len = len(llm.tokenize(suffix_text.encode("utf-8"), add_bos=False, special=True))
            if prefix_len + suffix_len > len(prompt_tokens):
                suffix_len = max(0, len(prompt_tokens) - prefix_len)

    if len(prompt_tokens) > prompt_limit:
        if prefix_len + suffix_len >= prompt_limit:
            raise ValueError(
                f"System/tool instructions and the latest user request need {prefix_len + suffix_len} tokens, "
                f"but only {prompt_limit} prompt tokens fit with an answer reserve. Reduce tool schemas or increase context."
            )
        middle_budget = prompt_limit - prefix_len - suffix_len
        middle_end = len(prompt_tokens) - suffix_len if suffix_len else len(prompt_tokens)
        middle = prompt_tokens[prefix_len:middle_end]
        kept_middle = middle[-middle_budget:] if middle_budget else []
        suffix = prompt_tokens[-suffix_len:] if suffix_len else []
        prompt_tokens = prompt_tokens[:prefix_len] + kept_middle + suffix
        print(
            f"Trimmed older conversation history to preserve system/tool instructions and reserve "
            f"at least {min_output} output tokens.",
            flush=True,
        )

    output_budget = min(max_output, max_ctx - len(prompt_tokens))
    if output_budget < min_output:
        raise ValueError(f"Only {output_budget} output tokens fit; at least {min_output} are reserved.")
    return prompt_tokens, output_budget


Trying n_ctx=42,000 ...
Model loaded on GPU with 42,000-token context.


In [9]:
import os
import time
import socket
import json
import re
import threading
import traceback
from pyngrok import ngrok
import gradio as gr
from fastapi import FastAPI, Request, Header, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from llama_cpp.llama_chat_format import Jinja2ChatFormatter
import uvicorn

# ---------------- 2. Configure Ngrok & Clean up previous runs ----------------
try:
    ngrok.set_auth_token(ngrok_token)
except Exception as e:
    if "Authtoken already set in configuration file" not in str(e):
        raise

# Terminate old tunnels and servers if re-running cell
try:
    ngrok.kill()
except Exception:
    pass

global uvicorn_server
if "uvicorn_server" in globals() and uvicorn_server is not None:
    uvicorn_server.should_exit = True
    time.sleep(1)

# ---------------- 3. Validate prerequisites ----------------
if "demo" not in globals():
    raise RuntimeError(
        "The variable 'demo' does not exist. Run the Gradio definition cell first."
    )

if "llm" not in globals():
    raise RuntimeError(
        "The variable 'llm' does not exist. Run your model-loading cell first."
    )

if "llm_lock" not in globals():
    llm_lock = threading.Lock()

In [10]:
# ---------------- 4. Define FastAPI with OpenAI Endpoints ----------------
api_app = FastAPI(
    title="Qwen 3.6 35B-A3B OpenAI Compatible Server",
    description="OpenAI-compatible endpoints for Continue (VS Code) & custom code editors",
    version="1.0.0"
)

api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_api_key(authorization: str = None, api_key_header: str = None, x_api_key: str = None, req_api_key: str = None):
    if not API_KEY:
        return True
    token = None
    if authorization:
        token = authorization.replace("Bearer ", "").strip()
    elif api_key_header:
        token = api_key_header.strip()
    elif x_api_key:
        token = x_api_key.strip()
    elif req_api_key:
        token = req_api_key.strip()

    if token != API_KEY:
        raise HTTPException(
            status_code=401,
            detail={
                "error": {
                    "message": "Invalid or missing API key. Provide header 'Authorization: Bearer YOUR_API_KEY'",
                    "type": "invalid_request_error",
                    "code": "invalid_api_key"
                }
            }
        )
    return True

def format_chat_prompt(messages, enable_thinking=False, tools=None):
    try:
        meta = llm.metadata
        eos = llm.detokenize([llm.token_eos()]).decode(errors="ignore")
        bos_id = llm.token_bos()
        bos = llm.detokenize([bos_id]).decode(errors="ignore") if bos_id >= 0 else ""
        tmpl = meta.get("tokenizer.chat_template")
        if tmpl:
            formatter = Jinja2ChatFormatter(
                template=tmpl,
                eos_token=eos, bos_token=bos, add_generation_prompt=True,
            )
            return formatter(
                messages=messages, tools=tools, enable_thinking=enable_thinking
            ).prompt
    except Exception as e:
        print(f"Jinja chat template warning: {e}, falling back to ChatML format")

    # Qwen3.6 ChatML fallback. Keep tool schemas and tool history when the
    # GGUF template is missing or incompatible with this llama.cpp build.
    system_messages = []
    conversation = []
    for message in messages:
        role = message.get("role", "user")
        if role in ("system", "developer"):
            text = str(message.get("content") or "").strip()
            if text:
                system_messages.append(text)
        else:
            conversation.append(message)

    if tools:
        tool_rows = [json.dumps(tool, ensure_ascii=False) for tool in tools]
        system_messages.insert(0, (
            "# Tools\n\n"
            "You have access to the following functions:\n\n"
            "<tools>\n" + "\n".join(tool_rows) + "\n</tools>\n\n"
            "If you choose to call a function, reply using this format and no suffix:\n\n"
            "<tool_call>\n"
            "<function=example_function_name>\n"
            "<parameter=example_parameter_name>\n"
            "value\n"
            "</parameter>\n"
            "</function>\n"
            "</tool_call>\n\n"
            "Required parameters must be specified. If no function is needed, "
            "answer normally without claiming to have used a function."
        ))

    parts = []
    if system_messages:
        parts.append(
            "<|im_start|>system\n"
            + "\n\n".join(system_messages)
            + "<|im_end|>\n"
        )

    for index, message in enumerate(conversation):
        role = message.get("role", "user")
        content = str(message.get("content") or "")

        if role == "user":
            parts.append(f"<|im_start|>user\n{content}<|im_end|>\n")
        elif role == "assistant":
            rendered = f"<|im_start|>assistant\n{content}"
            for call_index, tool_call in enumerate(message.get("tool_calls") or []):
                function = tool_call.get("function") or {}
                name = function.get("name")
                if not name:
                    raise ValueError("Assistant tool history is missing a function name.")
                arguments = function.get("arguments", {})
                if isinstance(arguments, str):
                    try:
                        arguments = json.loads(arguments) if arguments.strip() else {}
                    except json.JSONDecodeError:
                        arguments = {"_raw": arguments}
                if not isinstance(arguments, dict):
                    raise ValueError("Assistant tool history arguments must be an object.")
                if content or call_index:
                    rendered += "\n\n" if call_index == 0 else "\n"
                rendered += f"<tool_call>\n<function={name}>\n"
                for key, value in arguments.items():
                    value_text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False)
                    rendered += f"<parameter={key}>\n{value_text}\n</parameter>\n"
                rendered += "</function>\n</tool_call>"
            rendered += "<|im_end|>\n"
            parts.append(rendered)
        elif role == "tool":
            if index == 0 or conversation[index - 1].get("role") != "tool":
                parts.append("<|im_start|>user\n")
            parts.append(f"<tool_response>\n{content}\n</tool_response>")
            if index + 1 == len(conversation) or conversation[index + 1].get("role") != "tool":
                parts.append("<|im_end|>\n")
        else:
            raise ValueError(f"Unsupported message role in Qwen3.6 fallback: {role}")

    parts.append("<|im_start|>assistant\n")
    parts.append("<think>\n" if enable_thinking else "<think>\n\n</think>\n\n")
    return "".join(parts)
# ---------------- Tool-calling helpers (Qwen3.6 "qwen3_coder" XML tool-call format) ----------------
import uuid

TOOL_OPEN, TOOL_CLOSE = "<tool_call>", "</tool_call>"

def normalize_messages(raw_messages):
    # Keeps tool_calls / tool results so the chat template can render a full agent conversation.
    cleaned = []
    for m in raw_messages:
        c = m.get("content", "")
        if isinstance(c, list):
            parts = []
            for p in c:
                if isinstance(p, dict) and p.get("type") == "text":
                    parts.append(p.get("text", ""))
                elif isinstance(p, str):
                    parts.append(p)
            c = "\n".join(parts)
        msg = {"role": m.get("role", "user"), "content": str(c or "")}
        if m.get("tool_calls"):
            tcs = []
            for tc in m["tool_calls"]:
                fn = tc.get("function", {}) or {}
                args = fn.get("arguments", {})
                if isinstance(args, str):
                    try:
                        args = json.loads(args) if args.strip() else {}
                    except Exception:
                        args = {"_raw": args}
                tcs.append({"id": tc.get("id"), "type": "function",
                            "function": {"name": fn.get("name"), "arguments": args}})
            msg["tool_calls"] = tcs
        if m.get("tool_call_id"):
            msg["tool_call_id"] = m["tool_call_id"]
        if m.get("name"):
            msg["name"] = m["name"]
        cleaned.append(msg)
    return cleaned


def _compact_tool_schema(value):
    if isinstance(value, dict):
        compact = {}
        for key, item in value.items():
            if key == "description" and isinstance(item, str):
                summary = item.strip().splitlines()[0]
                if len(summary) > 180:
                    summary = summary[:179].rsplit(" ", 1)[0] + "…"
                compact[key] = summary
            else:
                compact[key] = _compact_tool_schema(item)
        return compact
    if isinstance(value, list):
        return [_compact_tool_schema(item) for item in value]
    return value

def select_prompt_tools(tools, messages):
    latest_user = next(
        (str(message.get("content") or "") for message in reversed(messages) if message.get("role") == "user"),
        "",
    )
    lower = latest_user.lower()
    browser_history = any(
        str(message.get("name", "")).startswith("mcp__playwright__")
        or any(
            str((call.get("function") or {}).get("name", "")).startswith("mcp__playwright__")
            for call in (message.get("tool_calls") or [])
        )
        for message in messages
    )
    asks_for_browser = browser_history or "playwright" in lower or (
        re.search(r"\b(browser|website|webpage|url|site)\b", lower)
        and re.search(r"\b(visit|open|view|browse|navigate|look|inspect)\b", lower)
    )
    if asks_for_browser:
        browser_tools = []
        for tool in tools:
            function = tool.get("function", tool) if isinstance(tool, dict) else {}
            if str(function.get("name", "")).startswith("mcp__playwright__"):
                browser_tools.append(tool)
        if browser_tools:
            interactive = re.search(r"\b(click|fill|submit|select|download|upload|login|press|scroll|hover)\b", lower)
            if not interactive:
                preferred = {
                    "mcp__playwright__browser_navigate",
                    "mcp__playwright__browser_snapshot",
                    "mcp__playwright__browser_wait_for",
                }
                selected = [
                    tool for tool in browser_tools
                    if (tool.get("function", tool) if isinstance(tool, dict) else {}).get("name") in preferred
                ]
                if selected:
                    return [_compact_tool_schema(tool) for tool in selected]
            return [_compact_tool_schema(tool) for tool in browser_tools]
    return []

def _visible_text(raw, enable_thinking=False):
    # Remove reasoning from model output (handles a template that pre-opens <think>).
    t = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)
    if "<think>" in t:
        t = t.split("<think>", 1)[0]
    elif "</think>" in t:
        t = t.split("</think>", 1)[1]
    elif enable_thinking:
        return ""
    return t.lstrip("\n")

def _coerce_param(value, schema_type=None):
    v = value
    if v.startswith("\n"):
        v = v[1:]
    if v.endswith("\n"):
        v = v[:-1]
    if schema_type == "string":
        return v
    try:
        return json.loads(v.strip())
    except Exception:
        return v

def parse_tool_calls(text, tools=None):
    # Returns (content_before_tool_calls, [OpenAI-style tool_call dicts]).
    schemas = {}
    for t in tools or []:
        fn = t.get("function", t) if isinstance(t, dict) else {}
        schemas[fn.get("name")] = (fn.get("parameters") or {}).get("properties", {}) or {}

    first = text.find(TOOL_OPEN)
    if first == -1:
        return text, []
    content = text[:first].rstrip()
    calls = []
    for block in re.findall(r"<tool_call>(.*?)(?:</tool_call>|\Z)", text[first:], flags=re.S):
        block = block.strip()
        name, args = None, {}
        m = re.search(r"<function=([^>\n]+)>(.*?)(?:</function>|\Z)", block, flags=re.S)
        if m:
            name = m.group(1).strip()
            props = schemas.get(name, {})
            for pm in re.finditer(r"<parameter=([^>\n]+)>(.*?)</parameter>", m.group(2), flags=re.S):
                key = pm.group(1).strip()
                ptype = (props.get(key) or {}).get("type")
                args[key] = _coerce_param(pm.group(2), ptype)
        else:
            try:  # JSON style fallback: {"name": ..., "arguments": {...}}
                obj = json.loads(block)
                name = obj.get("name")
                args = obj.get("arguments", {})
                if isinstance(args, str):
                    args = json.loads(args)
            except Exception:
                continue
        if name:
            calls.append({
                "id": f"call_{uuid.uuid4().hex[:24]}",
                "type": "function",
                "function": {"name": name, "arguments": json.dumps(args, ensure_ascii=False)},
            })
    return content, calls

@api_app.get("/health")
def health():
    return {"status": "healthy", "model": MODEL_ID}

@api_app.get("/v1/models")
@api_app.get("/models")
async def list_models(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    now = int(time.time())
    model_id = MODEL_ID
    return {
        "object": "list",
        "data": [
            {"id": model_id, "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None},
            {"id": "default", "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None}
        ]
    }

In [11]:
# ---------------- Chat completion streaming ----------------
def chat_stream_response(cmpl_id, created_ts, model_name, prompt_tokens, max_tokens, temperature, top_p, stop):
    # Streaming SSE generator with initial keepalive handshake
    def event_stream():
        try:
            # 1. Immediately send initial role chunk to open stream and prevent client timeout
            init_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"role": "assistant"}, "finish_reason": None}]
            }
            yield f"data: {json.dumps(init_chunk)}\n\n"

            with llm_lock:
                stream_iter = llm.create_completion(
                    prompt=prompt_tokens,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    stop=stop,
                    stream=True
                )

                in_think = False
                think_buf = ""

                for chunk in stream_iter:
                    text = chunk["choices"][0].get("text", "")
                    if not text:
                        continue

                    if in_think:
                        think_buf += text
                        if "</think>" in think_buf:
                            remainder = think_buf.split("</think>", 1)[1].lstrip("\n")
                            in_think = False
                            think_buf = ""
                            if remainder:
                                delta_chunk = {
                                    "id": cmpl_id,
                                    "object": "chat.completion.chunk",
                                    "created": created_ts,
                                    "model": model_name,
                                    "choices": [{"index": 0, "delta": {"content": remainder}, "finish_reason": None}]
                                }
                                yield f"data: {json.dumps(delta_chunk)}\n\n"
                        continue

                    if "<think>" in text:
                        pre, post = text.split("<think>", 1)
                        if pre:
                            delta_chunk = {
                                "id": cmpl_id,
                                "object": "chat.completion.chunk",
                                "created": created_ts,
                                "model": model_name,
                                "choices": [{"index": 0, "delta": {"content": pre}, "finish_reason": None}]
                            }
                            yield f"data: {json.dumps(delta_chunk)}\n\n"
                        if "</think>" in post:
                            remainder = post.split("</think>", 1)[1].lstrip("\n")
                            if remainder:
                                delta_chunk = {
                                    "id": cmpl_id,
                                    "object": "chat.completion.chunk",
                                    "created": created_ts,
                                    "model": model_name,
                                    "choices": [{"index": 0, "delta": {"content": remainder}, "finish_reason": None}]
                                }
                                yield f"data: {json.dumps(delta_chunk)}\n\n"
                        else:
                            in_think = True
                            think_buf = post
                        continue

                    # Normal stream token
                    delta_chunk = {
                        "id": cmpl_id,
                        "object": "chat.completion.chunk",
                        "created": created_ts,
                        "model": model_name,
                        "choices": [{"index": 0, "delta": {"content": text}, "finish_reason": None}]
                    }
                    yield f"data: {json.dumps(delta_chunk)}\n\n"

            # Final stop chunk
            stop_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(stop_chunk)}\n\n"

        except Exception as e:
            print("Error in streaming generation:", repr(e))
            traceback.print_exc()
            err_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"content": f"\n\n[Model Error: {e}]"}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(err_chunk)}\n\n"

        yield "data: [DONE]\n\n"

    return StreamingResponse(
        event_stream(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive", "X-Accel-Buffering": "no"}
    )


def chat_stream_response_tools(cmpl_id, created_ts, model_name, prompt_tokens, max_tokens, temperature, top_p, stop, tools, enable_thinking=False):
    # Streams normal text as it arrives, holds back "<tool_call>" blocks, then emits them as OpenAI tool_calls.
    def chunk(delta, finish=None):
        return "data: " + json.dumps({
            "id": cmpl_id, "object": "chat.completion.chunk", "created": created_ts, "model": model_name,
            "choices": [{"index": 0, "delta": delta, "finish_reason": finish}],
        }) + "\n\n"

    def event_stream():
        try:
            yield chunk({"role": "assistant"})
            raw, sent, finish_reason = "", 0, "stop"
            with llm_lock:
                for part in llm.create_completion(
                    prompt=prompt_tokens, max_tokens=max_tokens, temperature=temperature,
                    top_p=top_p, stop=stop, stream=True,
                ):
                    choice = part["choices"][0]
                    if choice.get("finish_reason"):
                        finish_reason = choice["finish_reason"]
                    text = choice.get("text", "")
                    if not text:
                        continue
                    raw += text
                    vis = _visible_text(raw, enable_thinking)
                    idx = vis.find(TOOL_OPEN)
                    if idx != -1:
                        safe = vis[:idx].rstrip()
                    else:
                        hold = 0
                        for k in range(min(len(TOOL_OPEN) - 1, len(vis)), 0, -1):
                            if vis.endswith(TOOL_OPEN[:k]):
                                hold = k
                                break
                        safe = vis[:len(vis) - hold]
                    if len(safe) > sent:
                        yield chunk({"content": safe[sent:]})
                        sent = len(safe)

            vis = _visible_text(raw, enable_thinking)
            content, calls = parse_tool_calls(vis, tools)
            if calls:
                if len(content) > sent:
                    yield chunk({"content": content[sent:]})
                for i, tc in enumerate(calls):
                    yield chunk({"tool_calls": [{"index": i, **tc}]})
                yield chunk({}, "tool_calls")
            else:
                if TOOL_OPEN in vis:
                    print("[tool-call] Model output contained an incomplete or unrecognized call; no tool was run.", flush=True)
                    yield chunk({"content": "[Tool call not executed: the model output was incomplete or not in the expected format.]"})
                elif len(vis) > sent:
                    yield chunk({"content": vis[sent:]})
                yield chunk({}, finish_reason)
        except Exception as e:
            print("Error in streaming generation:", repr(e))
            traceback.print_exc()
            yield chunk({"content": f"\n\n[Model Error: {e}]"}, "stop")
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        event_stream(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive", "X-Accel-Buffering": "no"}
    )


In [12]:
@api_app.post("/v1/completions")
@api_app.post("/completions")
async def text_completions(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    body = await request.json()

    prompt = body.get("prompt", "")
    temperature = float(body.get("temperature", 0.6))
    top_p = float(body.get("top_p", 0.9))
    stream = bool(body.get("stream", False))
    prompt_tokens = llm.tokenize(prompt.encode("utf-8"), add_bos=False, special=True)
    prompt_tokens, max_tokens = fit_prompt_to_context(
        prompt_tokens, prompt_text=prompt, requested_max_tokens=body.get("max_tokens")
    )

    stop = body.get("stop", ["<|im_end|>", "<|endoftext|>"])
    if isinstance(stop, str):
        stop = [stop, "<|im_end|>"]
    elif isinstance(stop, list):
        if "<|im_end|>" not in stop:
            stop.append("<|im_end|>")

    if not stream:
        with llm_lock:
            res = llm.create_completion(
                prompt=prompt_tokens,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                stop=stop,
                stream=False
            )
        return JSONResponse(res)

    def stream_prompt():
        with llm_lock:
            for chunk in llm.create_completion(
                prompt=prompt_tokens,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                stop=stop,
                stream=True
            ):
                yield f"data: {json.dumps(chunk)}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        stream_prompt(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive"}
    )

In [13]:
@api_app.post("/v1/chat/completions")
@api_app.post("/chat/completions")
async def chat_completions(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    body = await request.json()

    raw_messages = body.get("messages", [])
    model_name = body.get("model", MODEL_ID)
    temperature = float(body.get("temperature", 0.6))
    top_p = float(body.get("top_p", 0.9))
    stream = bool(body.get("stream", False))

    # Tool calling (OpenAI format). tool_choice="none" disables tools for this request.
    tools = body.get("tools") or None
    if body.get("tool_choice") == "none":
        tools = None
    enable_thinking = bool((body.get("chat_template_kwargs") or {}).get("enable_thinking", False))

    stop = body.get("stop", ["<|im_end|>", "<|endoftext|>"])
    if isinstance(stop, str):
        stop = [stop, "<|im_end|>"]
    elif isinstance(stop, list):
        if "<|im_end|>" not in stop:
            stop.append("<|im_end|>")

    # Normalize messages for tokenization (keeps assistant tool_calls and tool results)
    cleaned_messages = normalize_messages(raw_messages)

    cmpl_id = f"chatcmpl-{int(time.time()*1000)}"
    created_ts = int(time.time())

    # Render with the model's own chat template (tools are injected by the template)
    prompt_tools = select_prompt_tools(tools, cleaned_messages) if tools else None
    if tools:
        print(f"Prompting with {len(prompt_tools)} of {len(tools)} available tool schemas.", flush=True)
    prompt_text = format_chat_prompt(cleaned_messages, enable_thinking=enable_thinking, tools=prompt_tools)
    prompt_tokens = llm.tokenize(prompt_text.encode("utf-8"), add_bos=False, special=True)
    prompt_tokens, max_tokens = fit_prompt_to_context(
        prompt_tokens, prompt_text=prompt_text, requested_max_tokens=body.get("max_tokens")
    )

    if not stream:
        with llm_lock:
            res_iter = llm.create_completion(
                prompt=prompt_tokens,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                stop=stop,
                stream=False
            )
        full_text = _visible_text(res_iter["choices"][0]["text"], enable_thinking).strip()
        content, tool_calls = parse_tool_calls(full_text, tools) if tools else (full_text, [])
        message = {"role": "assistant", "content": content if (content or not tool_calls) else None}
        if tool_calls:
            message["tool_calls"] = tool_calls
        n_out = len(llm.tokenize(full_text.encode("utf-8")))
        return JSONResponse({
            "id": cmpl_id,
            "object": "chat.completion",
            "created": created_ts,
            "model": model_name,
            "choices": [{
                "index": 0,
                "message": message,
                "finish_reason": "tool_calls" if tool_calls else "stop"
            }],
            "usage": {
                "prompt_tokens": len(prompt_tokens),
                "completion_tokens": n_out,
                "total_tokens": len(prompt_tokens) + n_out
            }
        })

    if tools:
        return chat_stream_response_tools(
            cmpl_id=cmpl_id, created_ts=created_ts, model_name=model_name,
            prompt_tokens=prompt_tokens, max_tokens=max_tokens,
            temperature=temperature, top_p=top_p, stop=stop,
            tools=tools, enable_thinking=enable_thinking,
        )

    return chat_stream_response(
        cmpl_id=cmpl_id,
        created_ts=created_ts,
        model_name=model_name,
        prompt_tokens=prompt_tokens,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        stop=stop,
    )


In [14]:
from fastapi.routing import APIRoute

async def props():
    n_ctx = llm.n_ctx() if callable(llm.n_ctx) else llm.n_ctx
    return {
        "default_generation_settings": {"n_ctx": n_ctx, "params": {}},
        "total_slots": 1,
        "model_path": MODEL_ID,
        "modalities": {"vision": False, "audio": False},
        "chat_template": "",
        "build_info": "colab-llama-cpp-python",
    }

async def tools():
    return []

def add_front(path, fn):
    # drop any earlier registration, then put ours first so Gradio's "/" mount can't swallow it
    api_app.router.routes[:] = [r for r in api_app.router.routes if getattr(r, "path", None) != path]
    api_app.router.routes.insert(0, APIRoute(path, fn, methods=["GET"]))

add_front("/props", props)
add_front("/tools", tools)
print("routes added: /props, /tools")

routes added: /props, /tools


In [15]:
# ---------------- 5. Mount Gradio & Start Background Server ----------------
# Gradio is mounted at root ("/") so the web interface remains accessible at https://<domain>/
api_app = gr.mount_gradio_app(api_app, demo, path="/")

class BackgroundServer(uvicorn.Server):
    def install_signal_handlers(self):
        pass

sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()

print(f"Starting unified FastAPI + Gradio server on port {port}...")

config = uvicorn.Config(app=api_app, host="0.0.0.0", port=port, log_level="warning", loop="asyncio")
uvicorn_server = BackgroundServer(config=config)
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()

# Wait for server readiness
for _ in range(30):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=2):
            print("Unified server is ready.")
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Unified server failed to start.")

Starting unified FastAPI + Gradio server on port 42281...
Unified server is ready.


In [16]:
# ---------------- 6. Connect ngrok tunnel ----------------
try:
    tunnel = ngrok.connect(
        addr=port,
        proto="http",
        url="https://" + ngrok_domain
    )

    print("\n" + "=" * 65)
    print("🚀 SERVER ONLINE - GRADIO UI & OPENAI ENDPOINTS ACTIVE")
    print("=" * 65)
    print(f"🌐 Gradio Web UI        : {tunnel.public_url}")
    print(f"🤖 OpenAI API Base      : {tunnel.public_url}/v1")
    print(f"📦 Model Name in Config : {MODEL_ID}")
    print(f"🔑 API Key Status       : {'Active from Kaggle Secrets (kept confidential)' if API_KEY else 'No Key Set'}")
    print("=" * 65)
    print(f"""
To configure Continue (in VS Code ~/.continue/config.yaml):

name: Main Config
version: 1.0.0
schema: v1
models:
  - name: Qwen 3.6 35B-A3B
    provider: openai
    model: {MODEL_ID}
    apiBase: {tunnel.public_url}/v1
    apiKey: {API_KEY}
    roles:
      - chat
      - edit
      - apply
    defaultCompletionOptions:
      contextLength: {MODEL_CONTEXT_TOKENS}
    requestOptions:
      timeout: 300000
      headers:
        ngrok-skip-browser-warning: "1"
""")

except Exception as e:
    print("Ngrok tunnel failed:", repr(e))
    raise


🚀 SERVER ONLINE - GRADIO UI & OPENAI ENDPOINTS ACTIVE
🌐 Gradio Web UI        : https://cherub-yo-yo-daffodil.ngrok-free.dev
🤖 OpenAI API Base      : https://cherub-yo-yo-daffodil.ngrok-free.dev/v1
📦 Model Name in Config : qwen3.6-35b-a3b
🔑 API Key Status       : Active from Kaggle Secrets (kept confidential)

To configure Continue (in VS Code ~/.continue/config.yaml):

name: Main Config
version: 1.0.0
schema: v1
models:
  - name: Qwen 3.6 35B-A3B
    provider: openai
    model: qwen3.6-35b-a3b
    apiBase: https://cherub-yo-yo-daffodil.ngrok-free.dev/v1
    apiKey: heytherethisismyapikey
    roles:
      - chat
      - edit
      - apply
    defaultCompletionOptions:
      contextLength: 42000
    requestOptions:
      timeout: 300000
      headers:
        ngrok-skip-browser-warning: "1"



In [17]:
import urllib.request, json
# Self-test: local server + public URL
try:
    print("local /health :", urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=10).read().decode())
except Exception as e:
    print("local /health FAILED:", repr(e))
try:
    req = urllib.request.Request(tunnel.public_url + "/health", headers={"ngrok-skip-browser-warning": "1"})
    print("public /health:", urllib.request.urlopen(req, timeout=15).read().decode())
except Exception as e:
    print("public /health FAILED:", repr(e))

local /health : {"status":"healthy","model":"qwen3.6-35b-a3b"}
public /health: {"status":"healthy","model":"qwen3.6-35b-a3b"}


In [18]:
import urllib.request, json
base = tunnel.public_url
H = {"ngrok-skip-browser-warning": "1", "Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

def call(path, data=None):
    req = urllib.request.Request(base + path, data=json.dumps(data).encode() if data else None, headers=H)
    try:
        r = urllib.request.urlopen(req, timeout=120)
        return r.status, r.read().decode()[:300]
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()[:200]
    except Exception as e:
        return "ERR", repr(e)

print("Gradio  /          ", call("/")[0])
print("models  /v1/models ", call("/v1/models"))
print("props   /props     ", call("/props")[0])
print("chat    /v1/chat/completions:")
print(call("/v1/chat/completions", {"model": MODEL_ID, "messages": [{"role": "user", "content": "Say hi in 5 words."}], "max_tokens": 30}))

Gradio  /           200
models  /v1/models  (200, '{"object":"list","data":[{"id":"qwen3.6-35b-a3b","object":"model","created":1794691767,"owned_by":"kaggle-host","permission":[],"root":"qwen3.6-35b-a3b","parent":null},{"id":"default","object":"model","created":1794691767,"owned_by":"kaggle-host","permission":[],"root":"qwen3.6-35b-a3b","parent":nul')
props   /props      200
chat    /v1/chat/completions:
(200, '{"id":"chatcmpl-1794691767218","object":"chat.completion","created":1794691767,"model":"qwen3.6-35b-a3b","choices":[{"index":0,"message":{"role":"assistant","content":"Hi there, how are you today?"},"finish_reason":"stop"}],"usage":{"prompt_tokens":19,"completion_tokens":8,"total_tokens":27}}')
Prompting with 0 of 96 available tool schemas.
Prompting with 3 of 96 available tool schemas.
Prompting with 3 of 96 available tool schemas.
Prompting with 3 of 96 available tool schemas.
Prompting with 3 of 96 available tool schemas.
Prompting with 3 of 96 available tool schemas.
Prompti